# N 조건부 가치기저 M5의 배정·N 증분 대조군: Dunnhumby seed 42

완료된 M4와 actual M5는 재학습하지 않고 결과를 재사용합니다. 이 노트북은 다음 두 대조군만 새로 100 epoch 학습합니다.

1. **degree-matched N/V 순열:** 사용자 활동량 10분위 안에서 M2의 `q_N`, `q_V`, 유효 여부를 공동 순열합니다. M4의 실제 `q_C`와 개인별 양성가중은 유지합니다.
2. **V-only 상수 게이트:** 실제 `q_V`는 유지하고 N 조건부 게이트를 1.25로 고정합니다. 완료 actual M5에서 N 게이트가 사실상 1.25에 포화되었으므로, `q_N`의 사용자별 차등이 추가로 기여했는지 확인합니다.

단일 개발 seed의 기제 확인용이며 final test와 holdout은 만들지 않습니다. 두 경제 주지표에서 actual M5가 두 대조군을 모두 엄격히 이겨야 N과 V의 올바른 배정 후보로 판독합니다. 유의성·안정성·일반화는 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '08e0022881c822dc106e9c232c41c294025db27f'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_n_conditioned_value_basis_controls import (
    REUSED_MODEL_IDS,
    TRAINED_MODEL_IDS,
    configure_value_basis_controls,
    preflight_summary,
    run_value_basis_controls,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_value_basis_controls(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_n_conditioned_value_basis_controls_development_screen_v1',
    actual_result_json='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_n_conditioned_value_basis_reuse_controls_development_screen_v2/m5_n_conditioned_value_basis_bea6e1d8bf5b.json',
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == list(TRAINED_MODEL_IDS)
assert summary['reused_models'] == list(REUSED_MODEL_IDS)
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['m4_assignment'] == 'observed q_C and observed user-bin fit in every arm'
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['fixed']['one_training_loop_and_optimizer_per_arm'] is True
assert summary['fixed']['external_reranking'] is False
assert summary['fixed']['m3_edge_weight'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_value_basis_controls(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

main_metrics = [
    'model_id', 'role', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10',
    'vndcg@10', 'coverage@10',
]
print('1) 재사용 M4·actual M5와 새 대조군 핵심 절대지표')
show(result_df[[column for column in main_metrics if column in result_df.columns]])
print('2) 두 대조군 대비 actual M5 전체 지표')
show(result_df.attrs['comparison'])
print('3) actual·N/V 순열·V-only의 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) 순열 불변식')
print(json.dumps(result_df.attrs['control_diagnostics'], ensure_ascii=False, indent=2))
print('5) 기제 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('6) 재사용 결과 출처')
print(json.dumps(result_df.attrs['reference_provenance'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))